# NB03: Data-Driven Enrichment Analysis

Unbiased genome-wide enrichment of eggNOG orthologous groups (OGs) in plant-associated
vs non-plant species. Complements the literature-informed marker survey in NB02 by
discovering novel functional signatures without prior biological assumptions.

**Steps:**
1. Load NB01/NB02 outputs (species compartment labels, marker matrix)
2. Aggregate eggNOG OGs to species-level binary presence via Spark query
3. Fisher's exact test for each OG: plant-associated vs non-plant species, with BH-FDR correction
4. Phylum fixed-effects logistic regression for top-50 OGs to control for phylogenetic confounding
5. Volcano plot of enrichment results
6. Identify novel plant-associated markers (q < 0.05, OR > 2, surviving phylogenetic control)
7. Summary statistics and save all outputs

**Requires**: Spark (on BERDL JupyterHub)

**Outputs**: `data/species_og_matrix.csv`, `data/enrichment_results.csv`, `data/logistic_phylo_controlled.csv`, `data/novel_plant_markers.csv`, `figures/volcano_enrichment.png`

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    REPO = os.path.abspath(os.path.join(_here, '..', '..', '..'))
elif os.path.exists(os.path.join(_here, 'projects', 'plant_microbiome_ecotypes')):
    REPO = _here
else:
    REPO = os.path.abspath(os.path.join(_here, '..', '..', '..'))

PROJECT = os.path.join(REPO, 'projects', 'plant_microbiome_ecotypes')
DATA = os.path.join(PROJECT, 'data')
FIGURES = os.path.join(PROJECT, 'figures')

os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)

print(f'REPO: {REPO}')
print(f'DATA: {DATA}')

## 1. Load NB01/NB02 outputs

In [ ]:
# Load species compartment labels from NB01
species_comp = pd.read_csv(os.path.join(DATA, 'species_compartment.csv'))
print(f'Species compartment labels: {len(species_comp):,} species')
print(f'  Plant-associated: {species_comp["is_plant_associated"].sum():,}')
print(f'  Non-plant: {(species_comp["is_plant_associated"] == 0).sum():,}')

# Load species marker matrix from NB02
species_markers = pd.read_csv(os.path.join(DATA, 'species_marker_matrix.csv'))
print(f'\nSpecies marker matrix: {len(species_markers):,} species')

# Build lookup: species -> is_plant_associated, phylum
species_labels = species_comp[['gtdb_species_clade_id', 'is_plant_associated',
                                'dominant_compartment', 'phylum']].copy()
print(f'\nPhylum distribution (top 10):')
print(species_labels['phylum'].value_counts().head(10).to_string())

## 2. Aggregate eggNOG OGs to species-level

Query eggnog_mapper_annotations JOIN gene_cluster to get OG per species.
Binary presence: does species X have >= 1 gene cluster with OG Y?
Pre-filter: OGs present in >= 5% of species in at least one group (plant vs non-plant).

In [ ]:
og_matrix_path = os.path.join(DATA, 'species_og_matrix.csv')
if os.path.exists(og_matrix_path):
    species_og = pd.read_csv(og_matrix_path)
    print(f'Species-OG matrix (loaded from cache): {len(species_og):,} rows')
else:
    spark = get_spark_session()
    raw_og = spark.sql("""
        SELECT gc.gtdb_species_clade_id, ann.eggNOG_OGs
        FROM kbase_ke_pangenome.eggnog_mapper_annotations ann
        JOIN kbase_ke_pangenome.gene_cluster gc ON ann.query_name = gc.gene_cluster_id
        WHERE ann.eggNOG_OGs IS NOT NULL
    """).toPandas()
    print(f'Raw eggNOG annotations: {len(raw_og):,} rows')

    # Extract root-level OG: first entry before '@'
    # eggNOG_OGs format: "COG1073@1|root,COG1073@2|Bacteria,..."
    def extract_root_og(og_str):
        """Extract root-level OG ID from pipe-separated eggNOG_OGs string."""
        if pd.isna(og_str) or str(og_str).strip() == '':
            return None
        first_entry = str(og_str).split(',')[0]  # first OG entry
        og_id = first_entry.split('@')[0]         # part before '@'
        return og_id if og_id else None

    raw_og['og_id'] = raw_og['eggNOG_OGs'].apply(extract_root_og)
    raw_og = raw_og.dropna(subset=['og_id'])
    print(f'After OG extraction: {len(raw_og):,} rows')
    print(f'Unique OGs: {raw_og["og_id"].nunique():,}')

    # Binary presence: species X has >= 1 gene cluster with OG Y
    species_og = raw_og.groupby(['gtdb_species_clade_id', 'og_id']).size().reset_index(name='count')
    species_og['present'] = 1  # binary presence
    print(f'Species-OG pairs: {len(species_og):,}')

    # Pre-filter: OGs present in >= 5% of species in at least one group
    species_with_labels = species_og.merge(
        species_labels[['gtdb_species_clade_id', 'is_plant_associated']],
        on='gtdb_species_clade_id', how='inner'
    )

    n_plant = species_labels['is_plant_associated'].sum()
    n_nonplant = (species_labels['is_plant_associated'] == 0).sum()

    # Prevalence per OG per group
    plant_prev = species_with_labels[species_with_labels['is_plant_associated'] == 1].groupby('og_id').size() / n_plant
    nonplant_prev = species_with_labels[species_with_labels['is_plant_associated'] == 0].groupby('og_id').size() / n_nonplant

    # Keep OGs with >= 5% prevalence in at least one group
    prev_df = pd.DataFrame({'prev_plant': plant_prev, 'prev_nonplant': nonplant_prev}).fillna(0)
    keep_ogs = prev_df[(prev_df['prev_plant'] >= 0.05) | (prev_df['prev_nonplant'] >= 0.05)].index
    print(f'OGs passing 5% prevalence filter: {len(keep_ogs):,} / {raw_og["og_id"].nunique():,}')

    species_og = species_og[species_og['og_id'].isin(keep_ogs)].copy()
    species_og.to_csv(og_matrix_path, index=False)
    print(f'Saved species_og_matrix.csv: {len(species_og):,} rows')

print(f'\nSpecies in OG matrix: {species_og["gtdb_species_clade_id"].nunique():,}')
print(f'OGs in matrix: {species_og["og_id"].nunique():,}')

## 3. Fisher's exact test: plant-associated vs non-plant species

For each OG, construct a 2x2 contingency table and compute Fisher's exact test.
Apply Benjamini-Hochberg FDR correction across all OGs.

In [ ]:
# Merge species labels with OG presence
species_og_labeled = species_og.merge(
    species_labels[['gtdb_species_clade_id', 'is_plant_associated']],
    on='gtdb_species_clade_id', how='inner'
)

# Total species counts per group
all_plant_species = set(
    species_labels.loc[species_labels['is_plant_associated'] == 1, 'gtdb_species_clade_id']
)
all_nonplant_species = set(
    species_labels.loc[species_labels['is_plant_associated'] == 0, 'gtdb_species_clade_id']
)

# Restrict to species that appear in the OG matrix
species_in_og = set(species_og['gtdb_species_clade_id'].unique())
plant_species = all_plant_species & species_in_og
nonplant_species = all_nonplant_species & species_in_og

n_plant_total = len(plant_species)
n_nonplant_total = len(nonplant_species)
print(f'Plant species in OG matrix: {n_plant_total:,}')
print(f'Non-plant species in OG matrix: {n_nonplant_total:,}')

# Build sets: for each OG, which species have it?
og_species_plant = species_og_labeled[species_og_labeled['is_plant_associated'] == 1].groupby('og_id')['gtdb_species_clade_id'].apply(set).to_dict()
og_species_nonplant = species_og_labeled[species_og_labeled['is_plant_associated'] == 0].groupby('og_id')['gtdb_species_clade_id'].apply(set).to_dict()

# Run Fisher's exact test for each OG
all_ogs = species_og['og_id'].unique()
results = []

for og in all_ogs:
    plant_pos = len(og_species_plant.get(og, set()))
    plant_neg = n_plant_total - plant_pos
    nonplant_pos = len(og_species_nonplant.get(og, set()))
    nonplant_neg = n_nonplant_total - nonplant_pos

    # 2x2 contingency table
    table = [[plant_pos, plant_neg],
             [nonplant_pos, nonplant_neg]]
    odds_ratio, p_value = stats.fisher_exact(table)

    prev_plant = plant_pos / n_plant_total if n_plant_total > 0 else 0
    prev_nonplant = nonplant_pos / n_nonplant_total if n_nonplant_total > 0 else 0

    results.append({
        'og_id': og,
        'n_plant_pos': plant_pos,
        'n_plant_total': n_plant_total,
        'n_nonplant_pos': nonplant_pos,
        'n_nonplant_total': n_nonplant_total,
        'prev_plant': prev_plant,
        'prev_nonplant': prev_nonplant,
        'odds_ratio': odds_ratio,
        'p_value': p_value,
    })

enrichment = pd.DataFrame(results)

# BH-FDR correction
reject, q_values, _, _ = multipletests(enrichment['p_value'], method='fdr_bh')
enrichment['q_value'] = q_values

# log2(OR) with pseudocount for infinite values
enrichment['log2_or'] = np.log2(enrichment['odds_ratio'].replace({0: np.nan, np.inf: np.nan}))
# Cap extreme values for visualization
max_log2 = 10
enrichment.loc[enrichment['odds_ratio'] == np.inf, 'log2_or'] = max_log2
enrichment.loc[enrichment['odds_ratio'] == 0, 'log2_or'] = -max_log2

# Sort by q-value
enrichment = enrichment.sort_values('q_value').reset_index(drop=True)

print(f'\nTotal OGs tested: {len(enrichment):,}')
print(f'Significant (q < 0.05): {(enrichment["q_value"] < 0.05).sum():,}')
print(f'Enriched in plant (q<0.05, OR>1): {((enrichment["q_value"] < 0.05) & (enrichment["odds_ratio"] > 1)).sum():,}')
print(f'Depleted in plant (q<0.05, OR<1): {((enrichment["q_value"] < 0.05) & (enrichment["odds_ratio"] < 1)).sum():,}')

print(f'\nTop 20 plant-enriched OGs:')
top_enriched = enrichment[(enrichment['q_value'] < 0.05) & (enrichment['odds_ratio'] > 1)].head(20)
print(top_enriched[['og_id', 'prev_plant', 'prev_nonplant', 'odds_ratio', 'q_value', 'log2_or']].to_string(index=False))

In [ ]:
# Save enrichment results
enrichment.to_csv(os.path.join(DATA, 'enrichment_results.csv'), index=False)
print(f'Saved enrichment_results.csv: {len(enrichment):,} OGs')

## 4. Phylum fixed-effects logistic regression (H0_phylo test)

For the top-50 most significant OGs, fit a logistic regression:

    OG_present ~ is_plant + C(phylum)

This controls for phylogenetic confounding: an OG may appear enriched in
plant-associated species simply because certain phyla are overrepresented in
plant environments. Only phyla with >= 20 species are included.

In [ ]:
# Build species-level design matrix for logistic regression
design = species_labels[species_labels['gtdb_species_clade_id'].isin(species_in_og)].copy()

# Filter to phyla with >= 20 species
phylum_counts = design['phylum'].value_counts()
large_phyla = phylum_counts[phylum_counts >= 20].index.tolist()
design = design[design['phylum'].isin(large_phyla)].copy()
print(f'Species in design matrix (phyla >= 20 species): {len(design):,}')
print(f'Phyla retained: {len(large_phyla)}')
print(f'  ' + ', '.join(large_phyla[:10]) + ('...' if len(large_phyla) > 10 else ''))

# Top-50 OGs by q-value
top50_ogs = enrichment.head(50)['og_id'].tolist()
print(f'\nFitting logistic regression for {len(top50_ogs)} OGs...')

# Build presence matrix for top-50 OGs across design species
top50_presence = species_og[species_og['og_id'].isin(top50_ogs)].copy()
top50_pivot = top50_presence.pivot_table(
    index='gtdb_species_clade_id', columns='og_id',
    values='present', fill_value=0
).reset_index()

# Merge with design matrix
design_full = design.merge(top50_pivot, on='gtdb_species_clade_id', how='left')
for og in top50_ogs:
    if og in design_full.columns:
        design_full[og] = design_full[og].fillna(0).astype(int)

# Fit logistic regression for each OG
logistic_results = []

for og in top50_ogs:
    if og not in design_full.columns:
        continue

    # Rename column to safe variable name for formula
    safe_col = 'og_present'
    df_fit = design_full[['is_plant_associated', 'phylum', og]].copy()
    df_fit = df_fit.rename(columns={og: safe_col})

    # Skip if no variation in outcome
    if df_fit[safe_col].nunique() < 2:
        continue

    try:
        model = smf.logit(f'{safe_col} ~ is_plant_associated + C(phylum)', data=df_fit)
        result = model.fit(disp=0, maxiter=100, method='bfgs')

        if 'is_plant_associated' in result.params.index:
            coef = result.params['is_plant_associated']
            se = result.bse['is_plant_associated']
            pval = result.pvalues['is_plant_associated']
            or_val = np.exp(coef)
            ci_lo = np.exp(coef - 1.96 * se)
            ci_hi = np.exp(coef + 1.96 * se)

            logistic_results.append({
                'og_id': og,
                'coef_is_plant': coef,
                'se': se,
                'or_phylo_controlled': or_val,
                'ci_lo_95': ci_lo,
                'ci_hi_95': ci_hi,
                'p_value_phylo': pval,
                'n_species': len(df_fit),
                'converged': result.mle_retvals.get('converged', True),
            })
    except Exception as e:
        # Convergence failure or perfect separation — skip
        logistic_results.append({
            'og_id': og,
            'coef_is_plant': np.nan,
            'se': np.nan,
            'or_phylo_controlled': np.nan,
            'ci_lo_95': np.nan,
            'ci_hi_95': np.nan,
            'p_value_phylo': np.nan,
            'n_species': len(df_fit),
            'converged': False,
        })

logistic_df = pd.DataFrame(logistic_results)

# BH-FDR correction on phylo-controlled p-values
valid_mask = logistic_df['p_value_phylo'].notna()
if valid_mask.sum() > 0:
    _, q_phylo, _, _ = multipletests(
        logistic_df.loc[valid_mask, 'p_value_phylo'], method='fdr_bh'
    )
    logistic_df.loc[valid_mask, 'q_value_phylo'] = q_phylo
else:
    logistic_df['q_value_phylo'] = np.nan

print(f'\nLogistic regression results: {len(logistic_df)} OGs')
print(f'Converged: {logistic_df["converged"].sum()}')
n_survives = ((logistic_df['p_value_phylo'] < 0.05) & (logistic_df['or_phylo_controlled'] > 1)).sum()
print(f'Significant after phylo control (p < 0.05, OR > 1): {n_survives}')

print(f'\nTop results (sorted by p-value):')
print(logistic_df.dropna(subset=['p_value_phylo']).sort_values('p_value_phylo').head(15)[
    ['og_id', 'coef_is_plant', 'or_phylo_controlled', 'ci_lo_95', 'ci_hi_95',
     'p_value_phylo', 'q_value_phylo']
].to_string(index=False))

In [ ]:
# Save logistic regression results
logistic_df.to_csv(os.path.join(DATA, 'logistic_phylo_controlled.csv'), index=False)
print(f'Saved logistic_phylo_controlled.csv: {len(logistic_df)} OGs')

## 5. Volcano plot: log2(OR) vs -log10(q)

Color by significance:
- **Green**: enriched in plant (q < 0.05, OR > 2)
- **Red**: depleted in plant (q < 0.05, OR < 0.5)
- **Grey**: not significant

In [ ]:
# Known PGP / pathogenicity OG IDs to annotate if present
KNOWN_OGS = {
    # Nitrogen fixation
    'COG1348': 'NifH', 'COG2710': 'NifD', 'COG0348': 'NifK',
    # Type III secretion
    'COG1450': 'T3SS-SctN', 'COG4789': 'T3SS-SctC', 'COG1386': 'T3SS-SctV',
    # T6SS
    'COG3157': 'T6SS-VgrG', 'COG3521': 'T6SS-Hcp',
    # ACC deaminase
    'COG2515': 'AcdS',
    # Siderophore
    'COG1629': 'Siderophore',
    # Flagella
    'COG1344': 'Flagellin', 'COG1536': 'FlhA',
    # Chemotaxis
    'COG0643': 'CheA', 'COG0835': 'CheR',
}

fig, ax = plt.subplots(figsize=(10, 8))

# Compute -log10(q), capping at 50 for visualization
enrichment['neg_log10_q'] = -np.log10(enrichment['q_value'].clip(lower=1e-50))

# Classify points
sig_enriched = (enrichment['q_value'] < 0.05) & (enrichment['odds_ratio'] > 2)
sig_depleted = (enrichment['q_value'] < 0.05) & (enrichment['odds_ratio'] < 0.5)
ns = ~(sig_enriched | sig_depleted)

# Plot non-significant first (background)
ax.scatter(enrichment.loc[ns, 'log2_or'], enrichment.loc[ns, 'neg_log10_q'],
           c='#BDBDBD', s=8, alpha=0.4, label=f'NS ({ns.sum():,})', rasterized=True)

# Plot significant enriched (green)
ax.scatter(enrichment.loc[sig_enriched, 'log2_or'], enrichment.loc[sig_enriched, 'neg_log10_q'],
           c='#4CAF50', s=15, alpha=0.7, label=f'Enriched in plant ({sig_enriched.sum():,})',
           edgecolors='#2E7D32', linewidths=0.3)

# Plot significant depleted (red)
ax.scatter(enrichment.loc[sig_depleted, 'log2_or'], enrichment.loc[sig_depleted, 'neg_log10_q'],
           c='#F44336', s=15, alpha=0.7, label=f'Depleted in plant ({sig_depleted.sum():,})',
           edgecolors='#C62828', linewidths=0.3)

# Annotate known PGP/pathogenicity OGs if present
annotated_count = 0
for og_id, label in KNOWN_OGS.items():
    row = enrichment[enrichment['og_id'] == og_id]
    if len(row) > 0:
        x = row['log2_or'].values[0]
        y = row['neg_log10_q'].values[0]
        ax.annotate(label, (x, y), fontsize=7, fontweight='bold',
                    xytext=(5, 5), textcoords='offset points',
                    arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
                    color='black')
        annotated_count += 1

# Reference lines
ax.axhline(-np.log10(0.05), color='grey', linestyle='--', alpha=0.5, linewidth=0.8)
ax.axvline(np.log2(2), color='grey', linestyle='--', alpha=0.5, linewidth=0.8)
ax.axvline(-np.log2(2), color='grey', linestyle='--', alpha=0.5, linewidth=0.8)
ax.axvline(0, color='black', linestyle='-', alpha=0.2, linewidth=0.5)

ax.set_xlabel('log2(Odds Ratio)', fontsize=12)
ax.set_ylabel('-log10(q-value)', fontsize=12)
ax.set_title('Enrichment of eggNOG OGs: plant-associated vs non-plant species', fontsize=13)
ax.legend(loc='upper left', fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'volcano_enrichment.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved figures/volcano_enrichment.png')
print(f'Known PGP/pathogenicity OGs annotated: {annotated_count}')

## 6. Identify novel plant-associated markers

Novel markers are OGs that:
- Are significantly enriched in plant-associated species (q < 0.05, OR > 2)
- Survive phylogenetic control (logistic regression p < 0.05 after controlling for phylum)
- Are **not** in our curated known PGP/pathogenicity gene list from NB02

In [ ]:
# Merge Fisher enrichment with logistic regression results
combined = enrichment.merge(logistic_df[['og_id', 'or_phylo_controlled', 'p_value_phylo',
                                          'q_value_phylo', 'converged']],
                            on='og_id', how='left')

# Criteria for novel markers:
# 1. Fisher: q < 0.05 and OR > 2 (enriched in plant)
# 2. Phylo-controlled: p < 0.05 (survives after controlling for phylum)
# 3. Not a known PGP/pathogenicity OG
novel_mask = (
    (combined['q_value'] < 0.05)
    & (combined['odds_ratio'] > 2)
    & (combined['p_value_phylo'] < 0.05)
    & (~combined['og_id'].isin(KNOWN_OGS.keys()))
)

novel_markers = combined[novel_mask].copy()
novel_markers = novel_markers.sort_values('q_value').reset_index(drop=True)

print(f'Novel plant-associated markers: {len(novel_markers):,}')
print(f'  (Fisher q<0.05 & OR>2 & phylo-controlled p<0.05 & not known marker)')

if len(novel_markers) > 0:
    print(f'\nTop novel markers:')
    display_cols = ['og_id', 'prev_plant', 'prev_nonplant', 'odds_ratio', 'q_value',
                    'or_phylo_controlled', 'p_value_phylo']
    print(novel_markers[display_cols].head(20).to_string(index=False))
else:
    # Relax criteria: show OGs that pass Fisher but not phylo control
    fisher_only = combined[
        (combined['q_value'] < 0.05)
        & (combined['odds_ratio'] > 2)
        & (~combined['og_id'].isin(KNOWN_OGS.keys()))
    ]
    print(f'\nNo OGs survived both Fisher and phylo control.')
    print(f'OGs passing Fisher alone (q<0.05, OR>2, not known): {len(fisher_only):,}')
    if len(fisher_only) > 0:
        print(fisher_only[['og_id', 'prev_plant', 'prev_nonplant', 'odds_ratio',
                           'q_value']].head(10).to_string(index=False))

In [ ]:
# Save novel markers
novel_markers.to_csv(os.path.join(DATA, 'novel_plant_markers.csv'), index=False)
print(f'Saved novel_plant_markers.csv: {len(novel_markers)} markers')

## 7. Summary statistics and outputs

In [ ]:
print('=== NB03 Summary ===')
print(f'Species in OG matrix: {species_og["gtdb_species_clade_id"].nunique():,}')
print(f'  Plant-associated: {n_plant_total:,}')
print(f'  Non-plant: {n_nonplant_total:,}')
print(f'OGs tested: {len(enrichment):,}')
print(f'Significant OGs (q < 0.05): {(enrichment["q_value"] < 0.05).sum():,}')
print(f'  Enriched in plant (OR > 1): {((enrichment["q_value"] < 0.05) & (enrichment["odds_ratio"] > 1)).sum():,}')
print(f'  Depleted in plant (OR < 1): {((enrichment["q_value"] < 0.05) & (enrichment["odds_ratio"] < 1)).sum():,}')
print(f'  Strong enrichment (OR > 2): {((enrichment["q_value"] < 0.05) & (enrichment["odds_ratio"] > 2)).sum():,}')
print(f'  Strong depletion (OR < 0.5): {((enrichment["q_value"] < 0.05) & (enrichment["odds_ratio"] < 0.5)).sum():,}')
print(f'Logistic regression (top 50 OGs):')
print(f'  Converged: {logistic_df["converged"].sum()}')
print(f'  Surviving phylo control (p < 0.05): {(logistic_df["p_value_phylo"] < 0.05).sum()}')
print(f'Novel plant markers: {len(novel_markers):,}')
print(f'\nOutputs saved to {DATA}/')
print('  - species_og_matrix.csv')
print('  - enrichment_results.csv')
print('  - logistic_phylo_controlled.csv')
print('  - novel_plant_markers.csv')
print(f'\nFigures saved to {FIGURES}/')
print('  - volcano_enrichment.png')
print(f'\nReady for NB04 (compartment profiling)')